In [13]:
import pandas as pd
import sys
import os
# project root to path
sys.path.append(os.path.abspath(".."))
from src.data_cleaning import *
import src.data_quality as dq
from src.config import *

print(dir(dq))

['DataQualityReport', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [7]:
# Loading raw data
applications = pd.read_csv("../data/raw/applications.csv")
daily = pd.read_csv("../data/raw/daily_surveys.csv")
final = pd.read_csv("../data/raw/final_survey.csv")

print("Applications:", applications.shape)
print("Daily:", daily.shape)
print("Final:", final.shape)


Applications: (2450, 8)
Daily: (7143, 6)
Final: (1540, 5)


#### Data inspection

In [8]:
applications.head()

,student_id,year,country,gender,education_level,accepted,has_disability,disability_type
0,2024_0,2024,Kenya,non_binary,phd,1,0,none
1,2024_1,2024,India,non_binary,phd,0,0,none
2,2024_2,2024,Germany,non_binary,masters,0,1,motor
3,2024_3,2024,Peru,female,masters,0,0,none
4,2024_4,2024,Germany,female,masters,0,0,none


In [9]:
applications.info()

<class 'pandas.DataFrame'>
RangeIndex: 2450 entries, 0 to 2449
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   student_id       2450 non-null   str  
 1   year             2450 non-null   int64
 2   country          2450 non-null   str  
 3   gender           2450 non-null   str  
 4   education_level  2450 non-null   str  
 5   accepted         2450 non-null   int64
 6   has_disability   2450 non-null   int64
 7   disability_type  2450 non-null   str  
dtypes: int64(3), str(5)
memory usage: 153.3 KB


In [10]:
applications["country"].value_counts()

country
Peru              374
Kenya             351
India             346
Germany           337
UK                298
Brazil            292
USA               272
United Kingdom     65
Brasil             61
United States      31
US                 23
Name: count, dtype: int64

#### Data quality checks

In [11]:
app_report = validate_applications(applications)
daily_report = validate_daily(daily, applications)
final_report = validate_final(final, applications)

app_report.summary()
daily_report.summary()
final_report.summary()

NameError: name 'validate_applications' is not defined

#### Cleaning application dataset

In [ ]:
# standardise country names
applications = clean_countries(applications)

In [ ]:
#remove duplicates
applications = remove_duplicates(applications)

In [ ]:
#validate categories
applications["education_level"].value_counts()

In [ ]:
applications["education_level"] = applications["education_level"].str.lower()

#### cleaning daily survey dataset

In [ ]:
#checks missing values
daily.isnull().sum()
#drop missimg rows
daily = daily.dropna(subset=["student_id", "day", "engagement_score"])
#ensure valid ranges
daily = daily[
    (daily["engagement_score"] >= 1) &
    (daily["engagement_score"] <= 5)
]

#### Cleaning final survey dataset

In [ ]:
final = final.dropna(subset=["student_id", "completion"])
final["satisfaction"] = final["satisfaction"].clip(1, 5)
final["self_reported_learning"] = final["self_reported_learning"].clip(1, 5)

#### merge dataset

In [ ]:
# merge datasets

df = daily.merge(applications, on=["student_id", "year"], how="left")
df = df.merge(final, on=["student_id", "year"], how="left")

print("Merged shape:", df.shape)
df.head()

In [ ]:
# basic checks

print("Missing values after merge:\n")
print(df.isnull().sum())

In [ ]:
missing_students = df["student_id"].isnull().sum()
print("Missing student IDs after merge:", missing_students)

In [ ]:
# sanity checks after cleaning

df["country"].value_counts()

In [ ]:
df["disability_type"].value_counts()

In [ ]:
df["completion"].mean()

In [ ]:
#. save cleaned data
df.to_csv("../data/processed/merged_cleaned.csv", index=False)

In [ ]:
# run quality checks on cleaned data

app_report = validate_applications(applications)
app_report.summary()